# ArmanNN Training on Google Colab

This notebook trains ArmanNN using **pre-tokenized binary shards** for maximum GPU utilization.

## Workflow

1. **Prepare data once** — `prepare_data.py` downloads from HuggingFace, tokenizes, and writes binary `.bin` shards to disk
2. **Train from shards** — memory-mapped binary files feed the GPU at full speed with no network dependency

**Instructions:**
1. Go to Runtime → Change runtime type → Select **A100 GPU**
2. Run all cells in order
3. On subsequent runs, skip the data preparation cell (shards persist on disk)

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/mhd-rahman/Arman-NN.git
%cd Arman-NN

In [ ]:
!pip install -q torch datasets transformers numpy flash-attn --no-build-isolation

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Prepare pre-tokenized binary shards

This step downloads from HuggingFace, tokenizes with GPT-2 tokenizer, and writes binary `.bin` shard files.

**Run once** — if `data/pretrain/` already exists with shards, skip this cell.

Target mix (~12B tokens):
- FineWeb-Edu: 7.5B tokens (62.5%)
- Code (StarCoder): 1.5B tokens (12.5%)
- Wikipedia: 1.0B tokens (8.3%)
- OpenWebMath: 2.0B tokens (16.7%)

Storage: ~24 GB as uint16 binary files.

In [ ]:
import os
from pathlib import Path

# ============================================================
# DATA PREPARATION CONFIG
# ============================================================

DATA_ROOT = Path("data/pretrain")
TOKENIZER = "gpt2"
SEQ_LEN = 1024
SEED = 42

# Token targets per source
DATA_SOURCES = [
    {
        "name": "fineweb",
        "dataset": "HuggingFaceFW/fineweb-edu",
        "subset": "sample-10BT",
        "target_tokens": 7_500_000_000,
        "text_column": "text",
    },
    {
        "name": "code",
        "dataset": "bigcode/starcoderdata",
        "subset": None,
        "target_tokens": 1_500_000_000,
        "text_column": "content",
    },
    {
        "name": "wikipedia",
        "dataset": "wikimedia/wikipedia",
        "subset": "20231101.en",
        "target_tokens": 1_000_000_000,
        "text_column": "text",
    },
    {
        "name": "math",
        "dataset": "open-web-math/open-web-math",
        "subset": None,
        "target_tokens": 2_000_000_000,
        "text_column": "text",
    },
]


# ============================================================
# RUN PREPARATION (skip if shards already exist)
# ============================================================

from prepare_data import prepare_dataset, validate_shards

for source in DATA_SOURCES:
    output_dir = DATA_ROOT / source["name"]
    manifest_path = output_dir / "manifest.json"

    if manifest_path.exists():
        print(f"\n[SKIP] {source['name']}: shards already exist at {output_dir}")
        continue

    print(f"\n{'='*60}")
    print(f"Preparing: {source['name']} ({source['target_tokens']:,} tokens)")
    print(f"{'='*60}")

    prepare_dataset(
        dataset_name=source["dataset"],
        output_dir=output_dir,
        tokenizer_name=TOKENIZER,
        target_tokens=source["target_tokens"],
        shard_size=100_000_000,  # 100M tokens per shard (~200MB)
        text_column=source["text_column"],
        split="train",
        subset=source["subset"],
        streaming=True,
        seed=SEED,
        max_seq_len=SEQ_LEN,
    )

print("\n" + "="*60)
print("Data preparation complete!")
print("="*60)

# Show disk usage
total_size = sum(f.stat().st_size for f in DATA_ROOT.rglob("*.bin"))
print(f"Total shard storage: {total_size / 1e9:.1f} GB")

In [ ]:
# Optional: validate shard integrity
from prepare_data import validate_shards

for source in DATA_SOURCES:
    output_dir = DATA_ROOT / source["name"]
    print(f"\nValidating {source['name']}...")
    validate_shards(output_dir)

## 3. Load the dataset from binary shards

Memory-maps all shards for zero-copy random-access training. No HuggingFace connection needed from here on.

In [ ]:
import sys
sys.path.insert(0, ".")

import torch
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, ConcatDataset

from arman.training.data import BinaryShardDataset

# ============================================================
# CONFIG
# ============================================================

SEQ_LEN = 1024
DATA_ROOT = Path("data/pretrain")

# ============================================================
# LOAD ALL SOURCES AS A SINGLE CONCATENATED DATASET
# ============================================================

datasets = []
total_tokens = 0

for source_dir in sorted(DATA_ROOT.iterdir()):
    if not (source_dir / "manifest.json").exists():
        continue
    ds = BinaryShardDataset(source_dir, seq_len=SEQ_LEN)
    datasets.append(ds)
    total_tokens += ds.total_tokens
    print(f"  {source_dir.name:<12} {ds.total_tokens:>14,} tokens | {len(ds):>10,} sequences")

train_dataset = ConcatDataset(datasets)

print(f"\n{'='*60}")
print(f"Combined: {total_tokens:,} tokens | {len(train_dataset):,} sequences")
print(f"{'='*60}")

# ============================================================
# VALIDATION SET — Wikitext-2 test (fixed benchmark)
# ============================================================

EVAL_CACHE_PATH = "eval_dataset.pt"

class ListDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

import os
if os.path.exists(EVAL_CACHE_PATH):
    print(f"\nLoading cached eval dataset from {EVAL_CACHE_PATH}...")
    all_eval_samples = torch.load(EVAL_CACHE_PATH, weights_only=False)
    eval_dataset = ListDataset(all_eval_samples)
    print(f"Eval: {len(eval_dataset):,} sequences (loaded from cache)")
else:
    print("\nBuilding eval dataset (first time only, will cache to disk)...")
    from datasets import load_dataset
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained("gpt2")

    # Wikitext-2 test set (fixed benchmark)
    print("  Loading Wikitext-2 test...")
    eval_ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    wikitext_tokens = []
    for example in eval_ds:
        text = example["text"]
        if text.strip():
            wikitext_tokens.extend(tokenizer.encode(text, add_special_tokens=False))

    eval_samples = []
    for i in range(0, len(wikitext_tokens) - SEQ_LEN, SEQ_LEN):
        x = torch.tensor(wikitext_tokens[i:i + SEQ_LEN], dtype=torch.long)
        y = torch.tensor(wikitext_tokens[i + 1:i + SEQ_LEN + 1], dtype=torch.long)
        eval_samples.append((x, y))
    del wikitext_tokens, eval_ds

    # Also sample some sequences from the training shards as "in-domain" eval
    print("  Sampling in-domain eval from training shards...")
    import random
    random.seed(12345)
    in_domain_indices = random.sample(range(len(train_dataset)), min(7500, len(train_dataset)))
    in_domain_samples = [train_dataset[i] for i in in_domain_indices]

    all_eval_samples = eval_samples + in_domain_samples
    del eval_samples, in_domain_samples

    torch.save(all_eval_samples, EVAL_CACHE_PATH)
    print(f"  Saved eval cache to {EVAL_CACHE_PATH}")

    eval_dataset = ListDataset(all_eval_samples)
    print(f"\nTotal eval: {len(eval_dataset):,} sequences")

print("\nDatasets ready for training.")

## 4. Configure and build the model

In [ ]:
from arman.model import ArmanConfig, ArmanNN

config = ArmanConfig(
    vocab_size=50257,       # GPT-2 tokenizer vocab
    d_model=1024,
    n_layers=12,
    n_heads=16,
    max_seq_len=1024,
    mlp_hidden=2816,
    expert_hidden=1792,
    n_experts=4,
    moe_top_k=2,
    ssm_state_size=128,
    use_graph=False,        # Disabled — not feeding graph data during training
    use_memory=False,       # Disabled — adds noise without structured input
)

model = ArmanNN(config)
print(f"Parameters: {model.parameter_count():,}")

## 5. Train

Binary shards are memory-mapped — the DataLoader reads directly from disk at near-RAM speed. No tokenization or network calls during training.

In [ ]:
import logging
import time
from pathlib import Path

# Clear any stale GPU memory from previous runs
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Force logging output to display in notebook cells
log = logging.getLogger()
log.setLevel(logging.INFO)
log.handlers = []
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
log.addHandler(handler)

from arman.training.scheduler import get_cosine_schedule_with_warmup
from arman.training.checkpointing import save_checkpoint, load_checkpoint, find_latest_checkpoint
from arman.training.evaluator import Evaluator, EvalConfig

# ============================================================
# TRAINING HYPERPARAMETERS
# ============================================================

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
BATCH_SIZE = 32
GRAD_ACCUM = 1        # Effective batch = 32 sequences x 1024 tokens = ~32k tokens/step
WARMUP_STEPS = 0
TOTAL_STEPS = 76000
MIN_LR_RATIO = 0.1
SAVE_EVERY = 2000
EVAL_EVERY = 2000
LOG_EVERY = 50
CHECKPOINT_DIR = "checkpoints"

# ============================================================
# SETUP
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Device: {device}")

# Optimizer with weight decay separation
decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.ndim >= 2 and "norm" not in n]
no_decay_params = [p for n, p in model.named_parameters() if p.requires_grad and (p.ndim < 2 or "norm" in n)]
optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=LEARNING_RATE, betas=(0.9, 0.95))

# Resume from checkpoint if available
global_step = 0
ckpt = find_latest_checkpoint(CHECKPOINT_DIR)
if ckpt:
    info = load_checkpoint(ckpt, model, optimizer=optimizer, scheduler=None, device="cpu", strict=False)
    global_step = info["step"]
    print(f"Resumed from {ckpt} at step {global_step}")

scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP_STEPS, TOTAL_STEPS, MIN_LR_RATIO)
# Fast-forward scheduler to current step on resume
for _ in range(global_step):
    scheduler.step()

# bf16 — no GradScaler needed (more stable than fp16)
AMP_DTYPE = torch.bfloat16

# DataLoader — binary shards are random-access, so shuffle=True works great
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)

# ============================================================
# TRAINING LOOP
# ============================================================

model.train()
optimizer.zero_grad(set_to_none=True)
data_iter = iter(train_loader)
step_start = time.time()

effective_batch = BATCH_SIZE * GRAD_ACCUM
tokens_per_step = effective_batch * SEQ_LEN
print(f"Training for {TOTAL_STEPS} steps | effective batch = {effective_batch} | tokens/step = {tokens_per_step:,}")
print(f"Starting from step {global_step}")
print(f"Total training tokens: ~{TOTAL_STEPS * tokens_per_step / 1e9:.1f}B")

while global_step < TOTAL_STEPS:
    accum_loss = 0.0
    accum_aux = 0.0

    for _ in range(GRAD_ACCUM):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            x, y = next(data_iter)

        x, y = x.to(device), y.to(device)
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE):
            out = model(x, targets=y)
            loss = out["loss"] / GRAD_ACCUM

        loss.backward()
        accum_loss += out["loss"].item() / GRAD_ACCUM
        accum_aux += out["aux_loss"].item() / GRAD_ACCUM

    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()
    global_step += 1

    # Log
    if global_step % LOG_EVERY == 0:
        dt = time.time() - step_start
        lr = scheduler.get_last_lr()[0]
        tokens_sec = LOG_EVERY * tokens_per_step / dt
        print(f"step={global_step:06d} | loss={accum_loss:.4f} | aux={accum_aux:.4f} | "
              f"grad_norm={grad_norm:.3f} | lr={lr:.2e} | {tokens_sec/1e3:.1f}k tok/s | dt={dt:.2f}s")
        step_start = time.time()

    # Checkpoint
    if global_step % SAVE_EVERY == 0:
        ckpt_path = Path(CHECKPOINT_DIR) / f"step_{global_step:06d}.pt"
        save_checkpoint(ckpt_path, model, optimizer, scheduler, global_step, config)
        print(f"Saved checkpoint: {ckpt_path}")
        # Keep only last 2 checkpoints
        existing = sorted(Path(CHECKPOINT_DIR).glob("*.pt"))
        while len(existing) > 2:
            old = existing.pop(0)
            old.unlink()
            print(f"Removed old checkpoint: {old}")

    # Eval
    if global_step % EVAL_EVERY == 0:
        eval_cfg = EvalConfig(batch_size=16, use_amp=True, amp_dtype="bfloat16")
        evaluator = Evaluator(model=model, eval_config=eval_cfg, device=device)
        metrics = evaluator.evaluate(eval_dataset)
        print(f"[eval] step={global_step:06d} | {metrics}")
        model.train()

# Final save
ckpt_path = Path(CHECKPOINT_DIR) / f"step_{global_step:06d}.pt"
save_checkpoint(ckpt_path, model, optimizer, scheduler, global_step, config)
print(f"\nTraining complete! Final checkpoint: {ckpt_path}")

## 6. Evaluate the trained model

In [ ]:
from arman.training import Evaluator, EvalConfig

eval_config = EvalConfig(batch_size=16, use_amp=True, amp_dtype="bfloat16")
evaluator = Evaluator(model=model, eval_config=eval_config, device=device)

metrics = evaluator.evaluate(eval_dataset)
print("\n" + "=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  Loss:           {metrics.loss:.4f}")
print(f"  Perplexity:     {metrics.perplexity:.2f}")
print(f"  Top-1 Accuracy: {metrics.top1_accuracy*100:.2f}%")
print(f"  Top-5 Accuracy: {metrics.top5_accuracy*100:.2f}%")
print(f"  MRR:            {metrics.mrr:.4f}")
print("=" * 50)

## 7. Generate text from the trained model

In [ ]:
from transformers import AutoTokenizer
from generate import generate

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Encode a prompt
prompt = "The most important concept in machine learning is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# Generate
model.eval()
output_ids = generate(
    model,
    input_ids,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    top_p=0.9,
)

# Decode
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"Generated: {generated_text}")

## 8. Export to HuggingFace Hub

In [ ]:
# ============================================================
# Export ArmanNN to HuggingFace-compatible format & push to Hub
# ============================================================

import json
import os
from pathlib import Path
from safetensors.torch import save_file as save_safetensors
from huggingface_hub import HfApi, create_repo

# --- Config ---
HF_REPO_ID = "mhd-rahman/ArmanNN-Base"  # Change to your HF username/repo
EXPORT_DIR = "hf_export"
CHECKPOINT_PATH = None  # Set to a specific checkpoint, or None = use latest

# Find latest checkpoint if not specified
if CHECKPOINT_PATH is None:
    from arman.training.checkpointing import find_latest_checkpoint
    CHECKPOINT_PATH = find_latest_checkpoint("checkpoints")
    print(f"Using latest checkpoint: {CHECKPOINT_PATH}")

# Load checkpoint
state = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
model_state = state["model"]
config_dict = state["config"]
step = state["step"]
print(f"Loaded checkpoint at step {step}")

# Create export directory
export_path = Path(EXPORT_DIR)
export_path.mkdir(parents=True, exist_ok=True)

# 1. Save config.json (HF-style)
hf_config = {
    "architectures": ["ArmanNN"],
    "model_type": "arman-nn",
    "auto_map": {
        "AutoConfig": "configuration_arman.ArmanConfig",
        "AutoModelForCausalLM": "modeling_arman.ArmanForCausalLM",
    },
    **config_dict,
    "torch_dtype": "bfloat16",
    "training_step": step,
}
with open(export_path / "config.json", "w") as f:
    json.dump(hf_config, f, indent=2)
print("Saved config.json")

# 2. Save model weights as safetensors
save_state = {}
for k, v in model_state.items():
    if k == "lm_head.weight":
        continue  # Tied to token_embedding, skip
    save_state[f"model.{k}"] = v.contiguous().clone()
save_safetensors(save_state, str(export_path / "model.safetensors"))
print(f"Saved model.safetensors ({sum(p.numel() for p in save_state.values()):,} parameters)")
del save_state

# 3. Save generation_config.json
gen_config = {
    "max_new_tokens": 256,
    "temperature": 0.8,
    "top_k": 50,
    "top_p": 0.9,
    "repetition_penalty": 1.1,
    "do_sample": True,
}
with open(export_path / "generation_config.json", "w") as f:
    json.dump(gen_config, f, indent=2)
print("Saved generation_config.json")

# 4. Create configuration_arman.py (custom config for AutoConfig)
config_code = '''"""ArmanNN configuration for HuggingFace transformers."""
from transformers import PretrainedConfig

class ArmanConfig(PretrainedConfig):
    model_type = "arman-nn"

    def __init__(
        self,
        vocab_size=50257,
        d_model=1024,
        n_layers=12,
        n_heads=16,
        max_seq_len=1024,
        dropout=0.0,
        ssm_state_size=128,
        ssm_kernel_size=5,
        mlp_hidden=2816,
        n_experts=4,
        moe_top_k=2,
        expert_hidden=1792,
        graph_layers=2,
        memory_slots=128,
        memory_top_k=4,
        use_attention=True,
        use_ssm=True,
        use_mlp=True,
        use_moe=True,
        use_graph=False,
        use_memory=False,
        use_router=True,
        tie_embeddings=True,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.max_seq_len = max_seq_len
        self.dropout = dropout
        self.ssm_state_size = ssm_state_size
        self.ssm_kernel_size = ssm_kernel_size
        self.mlp_hidden = mlp_hidden
        self.n_experts = n_experts
        self.moe_top_k = moe_top_k
        self.expert_hidden = expert_hidden
        self.graph_layers = graph_layers
        self.memory_slots = memory_slots
        self.memory_top_k = memory_top_k
        self.use_attention = use_attention
        self.use_ssm = use_ssm
        self.use_mlp = use_mlp
        self.use_moe = use_moe
        self.use_graph = use_graph
        self.use_memory = use_memory
        self.use_router = use_router
        self.tie_embeddings = tie_embeddings
'''
with open(export_path / "configuration_arman.py", "w") as f:
    f.write(config_code)
print("Saved configuration_arman.py")

# 5. Create modeling_arman.py (custom model for AutoModelForCausalLM)
modeling_code = '''"""ArmanNN model for HuggingFace transformers."""
import sys
import os
from pathlib import Path

import torch
from transformers import PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

from .configuration_arman import ArmanConfig as HFArmanConfig

# Import the actual model code
sys.path.insert(0, str(Path(__file__).parent))
from arman.model.config import ArmanConfig as NativeConfig
from arman.model.model import ArmanNN


class ArmanForCausalLM(PreTrainedModel):
    config_class = HFArmanConfig
    supports_gradient_checkpointing = True
    _tied_weights_keys = ["model.lm_head.weight"]

    def __init__(self, config: HFArmanConfig):
        super().__init__(config)
        native_config = NativeConfig(
            vocab_size=config.vocab_size,
            d_model=config.d_model,
            n_layers=config.n_layers,
            n_heads=config.n_heads,
            max_seq_len=config.max_seq_len,
            dropout=config.dropout,
            ssm_state_size=config.ssm_state_size,
            mlp_hidden=config.mlp_hidden,
            n_experts=config.n_experts,
            moe_top_k=config.moe_top_k,
            expert_hidden=config.expert_hidden,
            graph_layers=config.graph_layers,
            memory_slots=config.memory_slots,
            memory_top_k=config.memory_top_k,
            use_attention=config.use_attention,
            use_ssm=config.use_ssm,
            use_mlp=config.use_mlp,
            use_moe=config.use_moe,
            use_graph=config.use_graph,
            use_memory=config.use_memory,
            use_router=config.use_router,
            tie_embeddings=config.tie_embeddings,
        )
        self.model = ArmanNN(native_config)

    def get_input_embeddings(self):
        return self.model.token_embedding

    def set_input_embeddings(self, value):
        self.model.token_embedding = value

    def get_output_embeddings(self):
        return self.model.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.model.lm_head = new_embeddings

    def forward(self, input_ids, attention_mask=None, labels=None, past_key_values=None, **kwargs):
        out = self.model(input_ids, targets=labels, past_key_values=past_key_values)
        return CausalLMOutputWithPast(
            loss=out["loss"],
            logits=out["logits"],
            past_key_values=out["present_key_values"],
        )

    def prepare_inputs_for_generation(self, input_ids, past_key_values=None, **kwargs):
        if past_key_values is not None:
            input_ids = input_ids[:, -1:]
        return {"input_ids": input_ids, "past_key_values": past_key_values}

    def _set_gradient_checkpointing(self, module, value=False):
        if value:
            self.model.enable_gradient_checkpointing()
        else:
            self.model.disable_gradient_checkpointing()
'''
with open(export_path / "modeling_arman.py", "w") as f:
    f.write(modeling_code)
print("Saved modeling_arman.py")

# 6. Copy the arman/ source package
import shutil
if (export_path / "arman").exists():
    shutil.rmtree(export_path / "arman")
shutil.copytree("arman", export_path / "arman")
print("Copied arman/ package")

# 7. Copy tokenizer files
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("gpt2")
tok.save_pretrained(str(export_path))
print("Saved tokenizer")

# 8. Create model card
model_card = f'''---
language: en
license: apache-2.0
tags:
  - arman-nn
  - hybrid-architecture
  - attention
  - ssm
  - moe
  - causal-lm
library_name: transformers
pipeline_tag: text-generation
---

# ArmanNN

A hybrid language model combining Causal Attention, Selective SSM (parallel scan),
Sparse Mixture-of-Experts, with learned fusion gates and path routers.

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("{HF_REPO_ID}")

inputs = tokenizer("The future of AI is", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Architecture

- **Parameters:** {sum(p.numel() for p in model_state.values()):,}
- **Layers:** {config_dict.get("n_layers", 12)}
- **d_model:** {config_dict.get("d_model", 1024)}
- **Heads:** {config_dict.get("n_heads", 16)}
- **Experts:** {config_dict.get("n_experts", 4)} (top-{config_dict.get("moe_top_k", 2)})
- **Max seq length:** {config_dict.get("max_seq_len", 1024)}
- **Trained steps:** {step:,}

## Training Data

Pre-tokenized binary shard corpus (~12B tokens):
- FineWeb-Edu (62.5%)
- Code - StarCoder (12.5%)
- Wikipedia (8.3%)
- OpenWebMath (16.7%)
'''
with open(export_path / "README.md", "w") as f:
    f.write(model_card)
print("Saved README.md (model card)")

print(f"\n{'='*50}")
print(f"Export complete! Files in: {EXPORT_DIR}/")
print(f"{'='*50}")

# 9. Push to HuggingFace Hub
PUSH_TO_HUB = True  # Set to False if you just want to export locally

if PUSH_TO_HUB:
    print(f"\nPushing to HuggingFace Hub: {HF_REPO_ID}")
    api = HfApi()
    create_repo(HF_REPO_ID, exist_ok=True)
    api.upload_folder(
        folder_path=EXPORT_DIR,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"Uploading raw checkpoint: {CHECKPOINT_PATH}")
    api.upload_file(
        path_or_fileobj=str(CHECKPOINT_PATH),
        path_in_repo=f"checkpoints/{Path(CHECKPOINT_PATH).name}",
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"Pushed! Model available at: https://huggingface.co/{HF_REPO_ID}")
    print(f"\nLoad with:")
    print(f"  model = AutoModelForCausalLM.from_pretrained(\"{HF_REPO_ID}\", trust_remote_code=True)")
    print(f"  tokenizer = AutoTokenizer.from_pretrained(\"{HF_REPO_ID}\")")